# Test the RCA orchestrator

This notebook is designed to run a **minimal end-to-end dry run** of the RCA orchestrator using pre-built JSON fixtures.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import os
import sys
from typing import Any, Dict, Optional

rca_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if rca_root not in sys.path:
    sys.path.insert(0, rca_root)

dackar_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if dackar_root not in sys.path:
    sys.path.insert(0, dackar_root)

from kg.py2neo_workflow import Py2Neo
from orchestrators.rca_reasoning_orchestrator import build_dev_orchestrator

In [2]:
EXTRACT_DIR = Path("fixtures/case_001_bearing_wear")
OUTPUT_DIR = Path("./rca_runs_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_DIR = Path("../schemas")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

VALIDATOR_MODE = "compat"
STOP_ON_VALIDATION_ERROR = False


## Utilities

In [3]:
def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def maybe_load_json(path: Path) -> Optional[Dict[str, Any]]:
    return load_json(path) if path.exists() else None


def normalize_event(event: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(event)
    if "event_id" not in out and "id" in out:
        out["event_id"] = out["id"]
    return out


def normalize_kg_context(kg_context: Dict[str, Any], event: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(kg_context)
    if "asset_id" not in out:
        asset_ids = ((out.get("seed_context") or {}).get("asset_ids") or [])
        out["asset_id"] = asset_ids[0] if asset_ids else event.get("asset_id")
    if "subgraph_id" not in out:
        out["subgraph_id"] = f"KGCTX::{event['event_id']}::{out.get('asset_id')}"
    return out


def normalize_candidates(cands: Dict[str, Any]) -> Dict[str, Any]:
    """
    Make older 4-dim fixtures compatible with the current engine/schema direction.
    """
    out = dict(cands)

    scoring_cfg = dict(out.get("scoring_config") or {})
    weights = dict(scoring_cfg.get("weights") or {})
    if "telemetry" not in weights:
        # Preserve original intent while making weights sum to 1.0
        weights = {
            "structural": 0.30,
            "temporal": 0.20,
            "telemetry": 0.20,
            "evidence": 0.20,
            "governance": 0.10,
        }
    scoring_cfg["weights"] = weights
    out["scoring_config"] = scoring_cfg

    fixed_candidates = []
    for c in out.get("candidates", []) or []:
        cc = dict(c)
        scores = dict(cc.get("scores") or {})
        if "telemetry" not in scores:
            # conservative backfill for old fixture
            scores["telemetry"] = scores.get("temporal", 0.5)
        cc["scores"] = scores

        if "score_rationale" not in cc:
            cc["score_rationale"] = {
                "structural": "Loaded from test fixture.",
                "temporal": "Loaded from test fixture.",
                "telemetry": "Backfilled for compatibility from temporal score.",
                "evidence": "Loaded from test fixture.",
                "governance": "Loaded from test fixture.",
            }

        if "temporal_evidence" not in cc:
            cc["temporal_evidence"] = {
                "tskr_rule_ids": [],
                "matching_signal_ids": [],
                "window_start": None,
                "window_end": None,
                "relation": "unknown",
                "operator_family": None,
                "mean_lag_hours": None,
                "support": None,
                "pattern_id": None,
            }

        fixed_candidates.append(cc)

    out["candidates"] = fixed_candidates
    return out


def derive_tskr_patterns(
    event: Dict[str, Any],
    telemetry_summary: Dict[str, Any],
    kg_context: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Minimal compatible temporal artifact for fixture-driven dry runs.
    """
    patterns = []
    anomaly_present = any((sig.get("anomalies") or []) for sig in telemetry_summary.get("signals", []) or [])
    for fm in kg_context.get("failure_modes", []) or []:
        fm_id = fm.get("fm_id")
        if not fm_id:
            continue
        patterns.append(
            {
                "pattern_id": f"TSKR::{fm_id}",
                "event_id": event["event_id"],
                "asset_id": event["asset_id"],
                "target_type": "failure_mode",
                "target_id": fm_id,
                "component_id": fm.get("component_id"),
                "relation": "simultaneous" if anomaly_present else "unknown",
                "operator_family": "interval_point",
                "mean_lag_hours": 0.0 if anomaly_present else None,
                "std_lag_hours": None,
                "support": 0.0,
                "confidence": 0.65 if anomaly_present else 0.0,
                "source": "test_rca_orchestrator.py",
            }
        )

    return {
        "event_id": event["event_id"],
        "asset_id": event["asset_id"],
        "patterns": patterns,
        "summary": {
            "has_temporal_support": bool(patterns),
            "mode": "fixture_stub",
            "n_patterns": len(patterns),
        },
        "provenance": {
            "generated_by": "test_rca_orchestrator.py",
            "generated_at": "2026-01-23T10:40:00Z",
        },
    }


def safe_get(d: Optional[Dict[str, Any]], *keys: str, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
    return default if cur is None else cur

## Load fixtures

In [4]:
event = normalize_event(load_json(EXTRACT_DIR / "event.json"))
telemetry_summary = load_json(EXTRACT_DIR / "telemetry_summary.json")
kg_context = normalize_kg_context(load_json(EXTRACT_DIR / "kg_context.json"), event)
causality_candidates = normalize_candidates(load_json(EXTRACT_DIR / "causality_candidates.json"))
evidence_bundle = load_json(EXTRACT_DIR / "evidence_bundle.json")
operational_context = maybe_load_json(EXTRACT_DIR / "operational_context.json")
pm_compliance = maybe_load_json(EXTRACT_DIR / "pm_compliance.json")
rca_card_seed = maybe_load_json(EXTRACT_DIR / "rca_card.json")
tskr_patterns = maybe_load_json(EXTRACT_DIR / "tskr_patterns.json")
if tskr_patterns is None:
    tskr_patterns = derive_tskr_patterns(event, telemetry_summary, kg_context)

assert event["asset_id"] == telemetry_summary["asset_id"]
assert kg_context["event_id"] == event["event_id"]
assert kg_context["asset_id"] == event["asset_id"]
assert causality_candidates.get("event_id") == event["event_id"]
assert evidence_bundle["retrieval_scope"]["asset_id"] == event["asset_id"]

print("Fixture sanity checks passed.")

Fixture sanity checks passed.


## Import orchestrator

In [5]:
client = Py2Neo(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

orchestrator = build_dev_orchestrator(
    output_dir=OUTPUT_DIR,
    client=client,
    database=NEO4J_DATABASE,
    schema_dir=SCHEMA_DIR,
    validator_mode=VALIDATOR_MODE,
    stop_on_validation_error=STOP_ON_VALIDATION_ERROR,
)

print("Orchestrator built.")

../schemas/operational_context.json
../schemas/evidence_bundle.json
../schemas/tskr_patterns.json
../schemas/telemetry_summary.json
../schemas/processed_text_record.json
../schemas/causality_candidates.json
../schemas/rca_card.json
../schemas/event.json
../schemas/pm_compliance.json
../schemas/kg_context.json
../schemas/document.json
Orchestrator built.


In [6]:
print(type(orchestrator.validator).__name__)
print(getattr(orchestrator.validator, "schema_dir", None))
print(sorted(getattr(orchestrator.validator, "schemas", {}).keys()))

RCAArtifactValidator
../schemas
['causality_candidates', 'document', 'event', 'evidence_bundle', 'kg_context', 'operational_context', 'pm_compliance', 'processed_text_record', 'rca_card', 'telemetry_summary', 'tskr_patterns']


## Run with prebuilt artifacts

In [7]:
try:
    result = orchestrator.run(
        event=event,
        telemetry_summary=telemetry_summary,
        operational_context=operational_context,
        pm_compliance=pm_compliance,
        kg_context=kg_context,
        tskr_patterns=tskr_patterns,
        causality_candidates=causality_candidates,
        evidence_bundle=evidence_bundle,
    )
finally:
    client.close()

print("Run completed.")
print("Returned keys:", sorted(result.keys()))

Run completed.
Returned keys: ['causality_candidates', 'evidence_bundle', 'input_validation', 'ishikawa_matrix', 'kg_context', 'output_validation', 'rca_card', 'run_context', 'run_manifest', 'tskr_patterns']


## Inspect outputs

In [8]:
for key in [
    "input_validation",
    "output_validation",
    "run_manifest",
    "kg_context",
    "tskr_patterns",
    "causality_candidates",
    "evidence_bundle",
    "rca_card",
]:
    if key in result:
        print(f"\n--- {key} ---")
        print(json.dumps(result[key], indent=2, default=str)[:4000])

summary = {
    "run_id": safe_get(result, "run_context", "run_id"),
    "event_id": safe_get(result, "rca_card", "event_id", default=event["event_id"]),
    "primary_hypothesis": safe_get(result, "rca_card", "primary_hypothesis", default={}),
    "n_candidates": len(safe_get(result, "causality_candidates", "candidates", default=[]) or []),
    "n_evidence": len(safe_get(result, "evidence_bundle", "results", default=[]) or []),
    "input_ok": safe_get(result, "input_validation", "ok"),
    "output_ok": safe_get(result, "output_validation", "ok"),
}

print("\n--- summary ---")
print(json.dumps(summary, indent=2))


--- input_validation ---
{
  "ok": true,
  "issues": [],
  "artifact": "bundle:inputs"
}

--- output_validation ---
{
  "ok": true,
  "issues": [],
  "artifact": "bundle:outputs"
}

--- run_manifest ---
{
  "run_id": "93a558f7-b479-4627-9270-8abab0ceea1a",
  "completed_at": "2026-03-26T19:00:10.808223+00:00",
  "input_refs": {
    "event_id": "E2026-01-23-001",
    "asset_id": "PUMP_A_01",
    "telemetry_asset_id": "PUMP_A_01",
    "has_operational_context": false,
    "has_pm_compliance": false
  },
  "artifacts": {
    "kg_context": {
      "present": true
    },
    "tskr_patterns": {
      "present": true,
      "pattern_count": 1
    },
    "causality_candidates": {
      "present": true,
      "candidate_count": 1
    },
    "evidence_bundle": {
      "present": true,
      "evidence_count": 1
    },
    "ishikawa_matrix": {
      "present": true
    },
    "rca_card": {
      "present": true
    }
  },
  "validation": {
    "inputs": {
      "ok": true,
      "issues": [],
    